# 📄 Research Paper Explainer Bot — RAG Pipeline
### Built Without Any Frameworks — Pure Python

---


| Part | What It Does |
|------|--------------|
| Setup + PDF Loading | Install libraries, load & clean PDF text |
| Text Chunking | Split text into overlapping chunks |
| Embeddings | Convert text to vectors using Sentence Transformers |
| FAISS Vector Search | Build index, search for similar chunks |
| LLM Answer Generation | Send context + query to Groq LLM |

---

## 🔁 Overall Pipeline (How the App Works)

```
PDF Upload
    ↓
[1] Extract raw text from PDF pages
    ↓
[2] Split text into overlapping chunks
    ↓
[3] Convert each chunk into a vector (embedding)
    ↓
[4] Store vectors in FAISS index
    ↓
User types a question
    ↓
[4] Embed the question → search FAISS → get top-K chunks
    ↓
[5] Send chunks + question to Groq LLM → get answer
```

This is called a **RAG (Retrieval-Augmented Generation)** pipeline.

---
## Setup + PDF Loading
**Responsible for:** Installing libraries and reading the PDF file

### Math/Concept Behind It:
No heavy math here — this is data ingestion.  
We use `PyPDF2` to read each page and join all the text.  
Then we clean it using regex: replace multiple spaces/newlines with a single space.

```
Raw PDF → Page1_text + Page2_text + ... → One big string → Cleaned string
```

The regex `\s+` matches any whitespace (spaces, tabs, newlines) and replaces with a single space.

In [1]:
# Install required libraries
!pip install -q PyPDF2 sentence-transformers faiss-cpu groq numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 10.3 MB/s eta 0:00:00


In [2]:
# --- Imports & Config ---

import PyPDF2
import numpy as np
import faiss
import re
import os
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from groq import Groq

# =============================================
# CONFIG — all settings in one place
# =============================================
CONFIG = {
    "chunk_size": 500,        # characters per chunk
    "chunk_overlap": 50,      # overlap between chunks
    "embedding_model": "all-MiniLM-L6-v2",   # small + fast
    "top_k": 3,               # how many chunks to retrieve
    "llm_model": "llama-3.1-8b-instant",      # Groq model
    "max_tokens": 500,
    "temperature": 0.7
}

print("✅ Libraries imported!")
print(f"Config: chunk_size={CONFIG['chunk_size']}, top_k={CONFIG['top_k']}")

✅ Libraries imported!
Config: chunk_size=500, top_k=3


In [3]:
# --- Set your Groq API Key ---

# Option A: If running on Google Colab with Secrets
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("✅ Groq API key loaded from Colab Secrets")
except Exception:
    # Option B: Paste your key directly (not recommended for sharing)
    os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
    print("⚠️  Using hardcoded key — don't share this notebook!")

✅ Groq API key loaded from Colab Secrets


In [5]:
# --- Load PDF ---

def load_pdf(pdf_path: str) -> str:
    """
    Reads every page of the PDF and combines into one string.
    Then cleans up extra spaces.
    """
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:                     # some pages may be images
                text += page_text + "\n"

    # Clean: collapse multiple whitespace into single space
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ⬇️  Upload your PDF to Colab and update the path
PDF_PATH = "/content/Attention_all_you_need.pdf"
pdf_text = load_pdf(PDF_PATH)

print(f"✅ PDF loaded! Total characters: {len(pdf_text)}")
print(f"\nFirst 300 characters preview:")
print(pdf_text[:300])

✅ PDF loaded! Total characters: 39483

First 300 characters preview:
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.comNoam Shazeer∗ Google Brain noam@google.comNiki Parma


---
## Text Chunking
**Responsible for:** Splitting the big text into smaller overlapping pieces

### 📐 The Math Behind Chunking

We can't feed the whole paper to the LLM at once (token limits!).  
So we split it into **chunks** of fixed size with some **overlap**.

**Why overlap?**  
If a sentence is cut in half at a chunk boundary, the overlap makes sure it appears fully in the next chunk too.

```
chunk_size = 500 characters
overlap    = 50 characters
step       = chunk_size - overlap = 450

Chunk 1: characters [0   → 499]
Chunk 2: characters [450 → 949]    ← 50 char overlap with chunk 1
Chunk 3: characters [900 → 1399]   ← 50 char overlap with chunk 2
...
```

**Total number of chunks ≈**
$$N = \left\lceil \frac{\text{len(text)}}{\text{chunk\_size} - \text{overlap}} \right\rceil$$

For a 36,000 char paper: $N \approx \lceil 36000 / 450 \rceil = 80$ chunks

In [6]:
# --- Chunk the Text ---

def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    """
    Splits text into chunks of `chunk_size` chars,
    with `overlap` chars shared between consecutive chunks.
    """
    chunks = []
    start = 0
    step = chunk_size - overlap     # how far we jump each time

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += step               # move forward by step (not full chunk_size)

    return chunks


chunks = chunk_text(pdf_text, CONFIG["chunk_size"], CONFIG["chunk_overlap"])

print(f"✅ Text split into {len(chunks)} chunks")
print(f"   Each chunk ≈ {CONFIG['chunk_size']} chars, overlap = {CONFIG['chunk_overlap']} chars")
print(f"\n--- Chunk 1 ---")
print(chunks[0])
print(f"\n--- Chunk 2 ---")
print(chunks[1])
print(f"\n🔎 Notice: last ~50 chars of Chunk 1 = first ~50 chars of Chunk 2 (overlap!)")

✅ Text split into 88 chunks
   Each chunk ≈ 500 chars, overlap = 50 chars

--- Chunk 1 ---
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.comNoam Shazeer∗ Google Brain noam@google.comNiki Parmar∗ Google Research nikip@google.comJakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.comAidan N. Gomez∗ † University of Toronto aidan@cs.toronto.eduŁukasz Kaise

--- Chunk 2 ---
ersity of Toronto aidan@cs.toronto.eduŁukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We p

---
## Sentence Embeddings
**Responsible for:** Converting text chunks into vectors

### 📐 The Math Behind Embeddings

A **sentence embedding** turns a piece of text into a list of numbers (a vector).  
Similar sentences → similar vectors.  
This is done by the pre-trained model `all-MiniLM-L6-v2`.

```
"The Transformer uses attention"  →  [-0.07, 0.05, 0.12, ..., 0.03]  (384 numbers)
"Attention mechanism in NLP"     →  [-0.06, 0.04, 0.11, ..., 0.04]  (384 numbers)
                                      ↑ These two are very close!
```

The model outputs vectors of size **384 dimensions**.  
So for N chunks, we get an **embedding matrix** of shape `(N, 384)`.

$$E = \begin{bmatrix} e_1 \\ e_2 \\ \vdots \\ e_N \end{bmatrix} \in \mathbb{R}^{N \times 384}$$

Each row $e_i$ is a vector representing one chunk of text.

In [7]:
# --- Create Embeddings ---

def create_embeddings(chunks: List[str], model_name: str):
    """
    Loads the embedding model and converts all chunks to vectors.
    Returns:
      embeddings  — numpy array of shape (num_chunks, 384)
      model       — the loaded model (reused later for query embedding)
    """
    print(f"Loading embedding model: {model_name}...")
    model = SentenceTransformer(model_name)

    print(f"Encoding {len(chunks)} chunks...")
    embeddings = model.encode(chunks)     # shape: (N, 384)

    return embeddings, model


embeddings, embedding_model = create_embeddings(chunks, CONFIG["embedding_model"])

print(f"\n✅ Embeddings created!")
print(f"   Shape: {embeddings.shape}  →  ({len(chunks)} chunks, 384 dimensions)")
print(f"\nSample (first 10 values of chunk 0):")
print(embeddings[0][:10])
print(f"\nMin value: {embeddings[0].min():.4f}  |  Max value: {embeddings[0].max():.4f}")

Loading embedding model: all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 88 chunks...

✅ Embeddings created!
   Shape: (88, 384)  →  (88 chunks, 384 dimensions)

Sample (first 10 values of chunk 0):
[ 0.10175855  0.00978408  0.00107573  0.05426374 -0.00140451 -0.00767711
 -0.03751116  0.00750345 -0.03503805  0.13308188]

Min value: -0.1438  |  Max value: 0.1517


---
### FAISS Vector Index + Retrieval
**Responsible for:** Building a searchable index and finding the most relevant chunks

### 📐 The Math Behind FAISS (L2 Distance Search)

FAISS = Facebook AI Similarity Search.  
We use `IndexFlatL2` which finds the **nearest vectors** using **Euclidean (L2) distance**.

**L2 Distance formula between two vectors A and B:**
$$d(A, B) = \sqrt{\sum_{i=1}^{384} (A_i - B_i)^2}$$

A smaller distance = more similar = more relevant chunk.

**How retrieval works step by step:**
```
1. User types: "What is attention mechanism?"
2. We embed this query → query_vec  (shape: 1 × 384)
3. FAISS computes L2 distance from query_vec to ALL chunk vectors
4. Returns the top_k=3 chunks with SMALLEST distance
```

**Example with simple 2D vectors:**
```
Query vec   = [0.5, 0.8]
Chunk A vec = [0.4, 0.7]  → d = √((0.5-0.4)² + (0.8-0.7)²) = √0.02 = 0.14  ← closest!
Chunk B vec = [0.9, 0.1]  → d = √((0.5-0.9)² + (0.8-0.1)²) = √0.65 = 0.81
```
FAISS picks Chunk A (distance 0.14).

In [8]:
# --- Build FAISS Index ---

def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatL2:
    """
    Builds a FAISS flat L2 index and adds all chunk embeddings.
    IndexFlatL2 = brute-force L2 distance search (exact, no approximation).
    """
    dimension = embeddings.shape[1]        # 384
    index = faiss.IndexFlatL2(dimension)   # create empty index
    index.add(embeddings.astype('float32'))# add all vectors
    return index


vector_index = build_faiss_index(embeddings)

print(f"✅ FAISS index built!")
print(f"   Vectors stored: {vector_index.ntotal}")
print(f"   Each vector has {embeddings.shape[1]} dimensions")

✅ FAISS index built!
   Vectors stored: 88
   Each vector has 384 dimensions


In [9]:
# --- Retrieve Relevant Chunks ---

def retrieve_chunks(query: str, embedding_model, vector_index, chunks: List[str], top_k: int) -> List[Dict]:
    """
    1. Embeds the query into a vector
    2. Searches FAISS for top_k nearest chunk vectors
    3. Returns those chunks + their L2 distances
    """
    # Step 1: Embed the query (shape: 1 × 384)
    query_vec = embedding_model.encode([query]).astype('float32')

    # Step 2: Search — returns distances and indices of nearest vectors
    distances, indices = vector_index.search(query_vec, top_k)
    # distances shape: (1, top_k)  |  indices shape: (1, top_k)

    # Step 3: Collect results
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "text": chunks[idx],
            "l2_distance": float(distances[0][i]),
            "chunk_id": int(idx)
        })

    return results


# Test retrieval
test_query = "What is the Transformer model?"
top_chunks = retrieve_chunks(test_query, embedding_model, vector_index, chunks, CONFIG["top_k"])

print(f"🔍 Query: '{test_query}'")
print(f"   Retrieved {len(top_chunks)} chunks\n")
for i, chunk in enumerate(top_chunks):
    print(f"--- Chunk {i+1} (Chunk ID: {chunk['chunk_id']}, L2 distance: {chunk['l2_distance']:.4f}) ---")
    print(chunk['text'][:200])
    print()

🔍 Query: 'What is the Transformer model?'
   Retrieved 3 chunks

--- Chunk 1 (Chunk ID: 15, L2 distance: 0.9172) ---
an input sequence of symbol representations (x1, ..., x n)to a sequence of continuous representations z= (z1, ..., z n). Given z, the decoder then generates an output sequence (y1, ..., y m)of symbols

--- Chunk 2 (Chunk ID: 50, L2 distance: 1.0616) ---
sults 6.1 Machine Translation On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big) in Table 2) outperforms the best previously reported models (including en

--- Chunk 3 (Chunk ID: 53, L2 distance: 1.1761) ---
es our results and compares our translation quality and training costs to other model architectures from the literature. We estimate the number of floating point operations used to train a model by mu



---
## LLM Answer Generation (Groq)
**Responsible for:** Sending the retrieved chunks + question to Groq LLM and getting the answer

### 📐 The Concept Behind RAG Answer Generation

This is the **Generation** part of RAG (Retrieval-Augmented Generation).

**Why don't we just ask the LLM directly?**  
LLMs can hallucinate — they make up facts.  
By giving the LLM only the retrieved chunks as context, we **ground** the answer in the actual paper.

**Prompt structure:**
```
System: "You are a research paper assistant. Answer only from the context."

User:
  Context:
    [Chunk 1]: ...
    [Chunk 2]: ...
    [Chunk 3]: ...

  Question: What is attention?

  Answer:
```

**Temperature** controls creativity:
- `temperature = 0.0` → always picks the most likely word (deterministic)
- `temperature = 1.0` → more random, creative
- We use `0.7` — a balance

Mathematically, temperature scales the probability distribution over next tokens:
$$P(\text{token}_i) = \frac{e^{\text{logit}_i / T}}{\sum_j e^{\text{logit}_j / T}}$$
where T = temperature.

In [10]:
# --- Generate Answer with Groq LLM ---

def generate_answer(query: str, relevant_chunks: List[Dict], config: Dict, client: Groq) -> Dict:
    """
    Builds a prompt with the retrieved chunks as context,
    then calls the Groq LLM to generate an answer.
    """
    # Combine all chunks into one context string
    context = "\n\n".join([
        f"[Chunk {i+1}]:\n{chunk['text']}"
        for i, chunk in enumerate(relevant_chunks)
    ])

    # Build the prompt
    prompt = f"""You are a helpful research paper assistant. Answer the question using ONLY the context below.

Context:
{context}

Question: {query}

Instructions:
- Answer ONLY from the context above
- If the context doesn't have enough info, say so
- Mention which chunk you used (e.g. "According to Chunk 1...")

Answer:"""

    # Call the Groq API
    try:
        response = client.chat.completions.create(
            model=config["llm_model"],
            messages=[
                {"role": "system", "content": "You are a research paper assistant."},
                {"role": "user",   "content": prompt}
            ],
            max_tokens=config["max_tokens"],
            temperature=config["temperature"]
        )
        return {
            "answer": response.choices[0].message.content,
            "chunks_used": len(relevant_chunks),
            "success": True
        }
    except Exception as e:
        return {"answer": f"Error: {e}", "chunks_used": 0, "success": False}


# Create Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("✅ Groq client ready!")

✅ Groq client ready!


---
## 🚀 Full Pipeline — Put It All Together

This is the complete flow that the Streamlit app runs every time a user asks a question.

In [11]:
# =============================================
# COMPLETE RAG PIPELINE — ASK ANY QUESTION
# =============================================

def ask(question: str):
    """
    Full pipeline:
      1. Embed the question
      2. Retrieve top-K relevant chunks from FAISS
      3. Send chunks + question to Groq LLM
      4. Print the answer
    """
    print(f"\n❓ Question: {question}")
    print("=" * 60)

    # Step 1 + 2: Retrieve
    relevant = retrieve_chunks(question, embedding_model, vector_index, chunks, CONFIG["top_k"])
    print(f"📦 Top {len(relevant)} chunks retrieved (L2 distances: {[round(c['l2_distance'],2) for c in relevant]})")

    # Step 3: Generate
    result = generate_answer(question, relevant, CONFIG, client)

    print(f"\n🤖 Answer:")
    print(result["answer"])
    print(f"\n📊 Chunks used: {result['chunks_used']} | Model: {CONFIG['llm_model']}")


# Try these questions!
ask("What is the Transformer model?")


❓ Question: What is the Transformer model?
📦 Top 3 chunks retrieved (L2 distances: [0.92, 1.06, 1.18])

🤖 Answer:
According to Chunk 1, the Transformer model architecture involves an input sequence of symbol representations (x1, ..., xn) being converted to a sequence of continuous representations z = (z1, ..., zn). The decoder then generates an output sequence (y1, ..., ym) of symbols one element at a time, using a process that is auto-regressive, consuming previously generated symbols as additional input when generating the next.

📊 Chunks used: 3 | Model: llama-3.1-8b-instant


In [12]:
ask("What is the role of self-attention?")


❓ Question: What is the role of self-attention?
📦 Top 3 chunks retrieved (L2 distances: [0.82, 1.01, 1.03])

🤖 Answer:
According to Chunk 1, self-attention is used to relate different positions of a single sequence in order to compute a representation of the sequence. It is also stated as an attention mechanism relating different positions of a single sequence, sometimes called intra-attention. 

Additionally, from Chunk 1, it is mentioned that self-attention has been used successfully in various tasks including reading comprehension and abstracting.

📊 Chunks used: 3 | Model: llama-3.1-8b-instant


In [13]:
ask("What datasets were used in the experiments?")


❓ Question: What datasets were used in the experiments?
📦 Top 3 chunks retrieved (L2 distances: [1.29, 1.48, 1.5])

🤖 Answer:
According to Chunk 2:

Two datasets were used in the experiments. The first one is the standard WMT 2014 English-German dataset consisting of about 4.5 million sentence pairs. The second one is the WMT 2014 English-French dataset, which is significantly larger, consisting of 36M sentences.

📊 Chunks used: 3 | Model: llama-3.1-8b-instant


---
## 📊 Bonus: Visualize What's Happening

This cell shows the L2 distances for different queries — a lower distance means a better match.

In [14]:
# Visual comparison: how distances change with different queries

queries = [
    "What is the Transformer?",
    "How does attention work?",
    "What is the weather today?"   # irrelevant — should have high distances
]

print("Query → L2 Distances of top-3 retrieved chunks")
print("=" * 55)
for q in queries:
    results = retrieve_chunks(q, embedding_model, vector_index, chunks, top_k=3)
    dists = [round(r['l2_distance'], 3) for r in results]
    avg = round(sum(dists) / len(dists), 3)
    print(f"\nQuery: '{q}'")
    print(f"  Distances: {dists}")
    print(f"  Avg L2 distance: {avg}  {'← small = relevant ✅' if avg < 1.0 else '← large = less relevant ⚠️'}")

Query → L2 Distances of top-3 retrieved chunks

Query: 'What is the Transformer?'
  Distances: [1.057, 1.25, 1.269]
  Avg L2 distance: 1.192  ← large = less relevant ⚠️

Query: 'How does attention work?'
  Distances: [0.856, 0.858, 0.862]
  Avg L2 distance: 0.859  ← small = relevant ✅

Query: 'What is the weather today?'
  Distances: [1.715, 1.813, 1.819]
  Avg L2 distance: 1.782  ← large = less relevant ⚠️


---
## 📝 Summary:

### The Whole Flow in One Line:
```
PDF → chunks → embeddings → FAISS index → (query) → top-K chunks → Groq LLM → Answer
```

This is **RAG: Retrieval-Augmented Generation** — grounding LLM answers in real documents.